In [1]:
# ============================================================
# CORRELATION ANALYSIS FOR Semnopithecus entellus
# SPECIE_04_PD0588
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import matplotlib.pyplot as plt

# ============================================================
# SETTINGS
# ============================================================

SPECIES = "Semnopithecus entellus"
SPECIES_SAFE = SPECIES.replace(" ", "_")

BASE_DIR = Path(r"C:\Users\ual-laptop\Desktop\CAPSTONE")

GBIF_CSV = BASE_DIR / "data" / "gbif" / "Semnopithecus entellus.csv"
CURRENT_DIR = BASE_DIR / "Time Period" / "Current" / "10min"

OUT_DIR = BASE_DIR / "SPECIE_04_PD0588_Semnopithecus_entellus" / "02_results" / "correlation"
OUT_DIR.mkdir(parents=True, exist_ok=True)

COMMON_BIO_IDS = [1,4,8,9,10,11,12,13,14,15,16,17,18,19]
CORR_THRESHOLD = 0.75
STUDY_BOUNDS = (67.12, -12.6, 124.12, 39.35)

# ============================================================
# FUNCTIONS
# ============================================================

def read_occurrences(csv_path):
    df = pd.read_csv(csv_path)

    lon_col = next((c for c in ["decimalLongitude","longitude","lon","x"] if c in df.columns), None)
    lat_col = next((c for c in ["decimalLatitude","latitude","lat","y"] if c in df.columns), None)

    df = df[[lon_col, lat_col]].dropna().copy()
    df.columns = ["lon","lat"]

    return gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.lon, df.lat), crs="EPSG:4326")

def clean_points(gdf):
    minx, miny, maxx, maxy = STUDY_BOUNDS

    gdf = gdf[
        (gdf.geometry.x >= minx) &
        (gdf.geometry.x <= maxx) &
        (gdf.geometry.y >= miny) &
        (gdf.geometry.y <= maxy)
    ].copy()

    gdf["x"] = gdf.geometry.x.round(6)
    gdf["y"] = gdf.geometry.y.round(6)
    gdf = gdf.drop_duplicates(subset=["x","y"])

    return gdf

def find_layers(folder, bio_ids):
    files = list(folder.glob("*.tif"))
    mapping = {}

    for bio in bio_ids:
        for f in files:
            if f"bio_{bio}" in f.name.lower():
                mapping[bio] = f
                break

    return mapping

def sample_values(layers, gdf):
    coords = [(p.x, p.y) for p in gdf.geometry]
    df = pd.DataFrame()

    for bio, path in layers.items():
        with rasterio.open(path) as src:
            vals = [v[0] for v in src.sample(coords)]
            vals = np.where(np.array(vals) == src.nodata, np.nan, vals)
            df[f"bio_{bio}"] = vals

    return df.dropna()

# ============================================================
# MAIN
# ============================================================

def main():
    print("Running correlation for:", SPECIES)

    occ = read_occurrences(GBIF_CSV)
    occ = clean_points(occ)

    print("Final occurrence points:", len(occ))

    layers = find_layers(CURRENT_DIR, COMMON_BIO_IDS)

    env = sample_values(layers, occ)

    corr = env.corr()

    # ========================
    # HIGH CORRELATION PAIRS
    # ========================
    pairs = []
    cols = corr.columns

    for i in range(len(cols)):
        for j in range(i+1, len(cols)):
            val = corr.iloc[i,j]
            if abs(val) > CORR_THRESHOLD:
                pairs.append([cols[i], cols[j], val])

    pairs_df = pd.DataFrame(pairs, columns=["Var1","Var2","Correlation"])

    # ========================
    # VARIABLE SELECTION
    # ========================
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

    drop = [c for c in upper.columns if any(upper[c] > CORR_THRESHOLD)]
    selected = [c for c in corr.columns if c not in drop]

    summary = pd.DataFrame({
        "Variable": corr.columns,
        "Status": ["Retained" if c in selected else "Dropped" for c in corr.columns]
    })

    # ========================
    # SAVE FILES
    # ========================

    corr.to_excel(OUT_DIR / f"{SPECIES_SAFE}_correlation_matrix.xlsx")
    pairs_df.to_excel(OUT_DIR / f"{SPECIES_SAFE}_high_correlation_pairs.xlsx", index=False)
    pd.DataFrame({"selected_variable": selected}).to_csv(
        OUT_DIR / f"{SPECIES_SAFE}_selected_variables.csv", index=False
    )
    summary.to_excel(OUT_DIR / f"{SPECIES_SAFE}_variable_selection_summary.xlsx", index=False)

    # ========================
    # HEATMAP
    # ========================

    fig, ax = plt.subplots(figsize=(12,10))
    im = ax.imshow(corr.values)

    ax.set_xticks(range(len(cols)))
    ax.set_yticks(range(len(cols)))
    ax.set_xticklabels(cols, rotation=45, ha="right")
    ax.set_yticklabels(cols)

    for i in range(len(cols)):
        for j in range(len(cols)):
            ax.text(j, i, f"{corr.iloc[i,j]:.2f}", ha="center", va="center", fontsize=7)

    plt.colorbar(im, ax=ax)
    plt.title(SPECIES + " Correlation Heatmap")

    plt.tight_layout()
    plt.savefig(OUT_DIR / f"{SPECIES_SAFE}_correlation_heatmap.png", dpi=300)
    plt.close()

    print("\nDONE ✅")
    print("Selected variables:", selected)
    print("Saved in:", OUT_DIR)

if __name__ == "__main__":
    main()

Running correlation for: Semnopithecus entellus
Final occurrence points: 258

DONE ✅
Selected variables: ['bio_1', 'bio_4', 'bio_8', 'bio_9', 'bio_10', 'bio_12', 'bio_14', 'bio_15', 'bio_18', 'bio_19']
Saved in: C:\Users\ual-laptop\Desktop\CAPSTONE\SPECIE_04_PD0588_Semnopithecus_entellus\02_results\correlation


In [5]:
# ============================================================
# FULL CORRECTED PAST SDM PIPELINE
# Semnopithecus entellus - SPECIE_04_PD0588
# Creates Current, LGM, Pleistocene, Pliocene + correct 4-panel
# ============================================================

import shutil
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.mask import mask
from rasterio.plot import plotting_extent
from shapely.geometry import box
import matplotlib.pyplot as plt

# ============================================================
# SETTINGS
# ============================================================

SPECIES = "Semnopithecus entellus"
SPECIES_SAFE = SPECIES.replace(" ", "_")

BASE_DIR = Path(r"C:\Users\ual-laptop\Desktop\CAPSTONE")
PROJECT_DIR = BASE_DIR / "SPECIE_04_PD0588_Semnopithecus_entellus"

OUT_DIR = PROJECT_DIR / "02_results" / "past"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CORR_DIR = PROJECT_DIR / "02_results" / "correlation"
SELECTED_CSV = CORR_DIR / f"{SPECIES_SAFE}_selected_variables.csv"

JAVA_EXE = Path(r"C:\Program Files\Java\jdk-17\bin\java.exe")
MAXENT_JAR = BASE_DIR / "maxent" / "maxent.jar"

GBIF_CSV = BASE_DIR / "data" / "gbif" / "Semnopithecus entellus.csv"
COUNTRIES_SHP = BASE_DIR / "data" / "boundaries" / "ne_10m_admin_0_countries" / "ne_10m_admin_0_countries.shp"

CURRENT_DIR = BASE_DIR / "Time Period" / "Current" / "10min"
LGM_DIR = BASE_DIR / "Time Period" / "Last Glacial Maximum" / "10min"
PLEISTOCENE_DIR = BASE_DIR / "Time Period" / "Pleistocene" / "10min"
PLIOCENE_DIR = BASE_DIR / "Time Period" / "Pliocene" / "10min"

STUDY_BOUNDS = (67.12, -12.6, 124.12, 39.35)

NODATA = -9999.0
FIG_DPI = 300
CMAP = "RdYlGn_r"

# ============================================================
# HELPERS
# ============================================================

def ensure_exists(path, label):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"{label} not found: {path}")

def remove_and_recreate_dir(path):
    path = Path(path)
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)

def load_selected_bio_ids():
    ensure_exists(SELECTED_CSV, "Selected variables CSV")

    df = pd.read_csv(SELECTED_CSV)
    col = "selected_variable" if "selected_variable" in df.columns else df.columns[0]

    bio_ids = []
    for v in df[col].dropna():
        v = str(v).strip().lower().replace("bio_", "").replace("bio", "")
        bio_ids.append(int(v))

    if not bio_ids:
        raise ValueError("No selected BIO layers found.")

    print("Using selected BIO layers:", bio_ids)
    return bio_ids

def read_occurrences():
    df = pd.read_csv(GBIF_CSV)

    lon_col = next((c for c in ["decimalLongitude", "longitude", "lon", "x"] if c in df.columns), None)
    lat_col = next((c for c in ["decimalLatitude", "latitude", "lat", "y"] if c in df.columns), None)

    if lon_col is None or lat_col is None:
        raise ValueError(f"Longitude/latitude columns not found. Columns: {list(df.columns)}")

    df = df[[lon_col, lat_col]].dropna().copy()
    df.columns = ["lon", "lat"]

    gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df["lon"], df["lat"]),
        crs="EPSG:4326"
    )

    minx, miny, maxx, maxy = STUDY_BOUNDS
    gdf = gdf[
        (gdf.geometry.x >= minx) &
        (gdf.geometry.x <= maxx) &
        (gdf.geometry.y >= miny) &
        (gdf.geometry.y <= maxy)
    ].copy()

    gdf["x_round"] = gdf.geometry.x.round(6)
    gdf["y_round"] = gdf.geometry.y.round(6)
    gdf = gdf.drop_duplicates(subset=["x_round", "y_round"]).copy()
    gdf = gdf.drop(columns=["x_round", "y_round"])

    print("Clean occurrence points:", len(gdf))
    return gdf

def export_samples(points_gdf):
    samples_csv = OUT_DIR / f"{SPECIES_SAFE}_maxent_samples.csv"

    pd.DataFrame({
        "species": SPECIES,
        "longitude": points_gdf.geometry.x,
        "latitude": points_gdf.geometry.y
    }).to_csv(samples_csv, index=False)

    return samples_csv

def build_study_polygon():
    return gpd.GeoDataFrame(
        geometry=[box(*STUDY_BOUNDS)],
        crs="EPSG:4326"
    )

def load_countries():
    countries = gpd.read_file(COUNTRIES_SHP)
    if countries.crs is None:
        countries = countries.set_crs("EPSG:4326")
    return countries.to_crs("EPSG:4326")

def find_bio_layers(folder, bio_ids):
    files = list(Path(folder).glob("*.tif"))

    if not files:
        raise FileNotFoundError(f"No .tif files found in {folder}")

    mapping = {}

    for bio_id in bio_ids:
        candidates = []

        for f in files:
            stem = f.stem.lower().replace("-", "_")

            if (
                stem == f"bio_{bio_id}"
                or stem == f"bio{bio_id}"
                or stem == f"bio{bio_id:02d}"
                or f"bio_{bio_id}" in stem
                or f"bio{bio_id:02d}" in stem
                or f"bio{bio_id}" in stem
            ):
                candidates.append(f)

        if not candidates:
            raise FileNotFoundError(f"Could not find BIO{bio_id} in {folder}")

        mapping[bio_id] = sorted(candidates)[0]

    return mapping

def crop_raster(src_path, out_path, region):
    with rasterio.open(src_path) as src:
        img, transform = mask(
            src,
            list(region.geometry),
            crop=True,
            filled=False
        )

        # Important fix: convert to float before filling with -9999
        arr = np.ma.array(img[0], copy=False).astype("float32")
        arr = arr.filled(NODATA).astype("float32")
        arr = np.where(np.isfinite(arr), arr, NODATA).astype("float32")

        meta = src.meta.copy()
        meta.update({
            "driver": "GTiff",
            "count": 1,
            "height": arr.shape[0],
            "width": arr.shape[1],
            "transform": transform,
            "dtype": "float32",
            "nodata": NODATA
        })

        out_path.parent.mkdir(parents=True, exist_ok=True)

        with rasterio.open(out_path, "w", **meta) as dst:
            dst.write(arr, 1)

def tif_to_ascii(src_tif, out_asc):
    with rasterio.open(src_tif) as src:
        band = src.read(1, masked=True).astype("float32")

        arr = np.ma.array(band, copy=False).filled(NODATA).astype("float32")
        arr = np.where(np.isfinite(arr), arr, NODATA).astype("float32")

        transform = src.transform
        nrows, ncols = arr.shape

        xll = transform.c
        yll = transform.f + transform.e * nrows
        cellsize = transform.a

        out_asc.parent.mkdir(parents=True, exist_ok=True)

        with open(out_asc, "w") as f:
            f.write(f"ncols        {ncols}\n")
            f.write(f"nrows        {nrows}\n")
            f.write(f"xllcorner    {xll}\n")
            f.write(f"yllcorner    {yll}\n")
            f.write(f"cellsize     {cellsize}\n")
            f.write(f"NODATA_value {NODATA}\n")

            for row in arr:
                f.write(" ".join(f"{float(v):.6f}" for v in row) + "\n")

def validate_ascii_dir(folder):
    asc_files = list(Path(folder).glob("*.asc"))
    if not asc_files:
        raise ValueError(f"No ASCII files found in {folder}")

def run_maxent(samples_csv, env_dir, projection_dir, out_dir):
    validate_ascii_dir(env_dir)
    validate_ascii_dir(projection_dir)

    remove_and_recreate_dir(out_dir)

    cmd = [
        str(JAVA_EXE),
        "-Xmx4g",
        "-jar",
        str(MAXENT_JAR),
        f"samplesfile={samples_csv}",
        f"environmentallayers={env_dir}",
        f"projectionlayers={projection_dir}",
        f"outputdirectory={out_dir}",
        "outputformat=cloglog",
        "randomtestpoints=25",
        "maximumbackground=10000",
        "maximumiterations=5000",
        "autorun",
        "redoifexists",
        "novisible",
        "pictures=false",
        "responsecurves=false",
        "jackknife=false",
        "writebackgroundpredictions=false"
    ]

    print("\nRunning MaxEnt:")
    print(" ".join(cmd))

    result = subprocess.run(cmd, capture_output=True, text=True)

    print(result.stdout)
    print(result.stderr)

    if result.returncode != 0:
        raise RuntimeError("MaxEnt failed. Check output above.")

def find_prediction(folder, keyword=None):
    files = list(Path(folder).rglob("*.asc"))

    files = [
        f for f in files
        if "clamping" not in f.name.lower()
        and "novel" not in f.name.lower()
        and "limiting" not in f.name.lower()
        and "background" not in f.name.lower()
    ]

    if keyword:
        selected = [f for f in files if keyword in f.name.lower()]
        if selected:
            files = selected

    if not files:
        raise FileNotFoundError(f"No prediction raster found in {folder}")

    chosen = sorted(
        files,
        key=lambda p: (
            "median" not in p.name.lower(),
            "avg" not in p.name.lower(),
            p.name.lower()
        )
    )[0]

    print("Prediction chosen:", chosen)
    return chosen

def load_raster(path):
    with rasterio.open(path) as src:
        arr = src.read(1).astype(float)

        nodata = src.nodata
        if nodata is not None:
            arr[arr == nodata] = np.nan

        arr[np.isinf(arr)] = np.nan
        extent = plotting_extent(src)

    return arr, extent

def get_boundaries(countries):
    region = box(*STUDY_BOUNDS)
    region_gdf = gpd.GeoDataFrame(geometry=[region], crs="EPSG:4326")

    clipped = countries[countries.intersects(region)].copy()
    if clipped.empty:
        return clipped

    return gpd.clip(clipped, region_gdf)

def plot_map(raster_path, countries, title, out_png):
    arr, extent = load_raster(raster_path)

    finite = arr[np.isfinite(arr)]
    if finite.size == 0:
        raise ValueError(f"No valid values in {raster_path}")

    vmin = np.percentile(finite, 2)
    vmax = np.percentile(finite, 98)

    if not np.isfinite(vmin) or not np.isfinite(vmax) or vmin == vmax:
        vmin = np.nanmin(finite)
        vmax = np.nanmax(finite)

    minx, miny, maxx, maxy = STUDY_BOUNDS
    boundaries = get_boundaries(countries)

    fig, ax = plt.subplots(figsize=(10, 6))

    im = ax.imshow(
        arr,
        extent=extent,
        origin="upper",
        cmap=CMAP,
        vmin=vmin,
        vmax=vmax
    )

    if not boundaries.empty:
        boundaries.boundary.plot(ax=ax, color="black", linewidth=0.5)

    ax.set_xlim(minx, maxx)
    ax.set_ylim(miny, maxy)

    ax.set_title(title, fontsize=14)
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")

    cbar = plt.colorbar(im, ax=ax, fraction=0.03, pad=0.03)
    cbar.set_label("Habitat suitability")

    plt.tight_layout()
    plt.savefig(out_png, dpi=FIG_DPI, bbox_inches="tight")
    plt.close()

def make_four_panel(pngs, final_png):
    items = [
        ("Current", pngs["current"]),
        ("Last Glacial Maximum", pngs["lgm"]),
        ("Pleistocene", pngs["pleistocene"]),
        ("Pliocene", pngs["pliocene"]),
    ]

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.ravel()

    for ax, (title, img_path) in zip(axes, items):
        ax.imshow(plt.imread(img_path))
        ax.axis("off")
        ax.set_title(title, fontsize=12)

    fig.suptitle(SPECIES, fontsize=18)
    plt.tight_layout()
    plt.savefig(final_png, dpi=FIG_DPI, bbox_inches="tight")
    plt.close()

# ============================================================
# MAIN
# ============================================================

def main():
    ensure_exists(SELECTED_CSV, "Selected variables CSV")
    ensure_exists(JAVA_EXE, "java.exe")
    ensure_exists(MAXENT_JAR, "maxent.jar")
    ensure_exists(GBIF_CSV, "GBIF CSV")
    ensure_exists(COUNTRIES_SHP, "Countries shapefile")
    ensure_exists(CURRENT_DIR, "Current climate folder")
    ensure_exists(LGM_DIR, "Last Glacial Maximum folder")
    ensure_exists(PLEISTOCENE_DIR, "Pleistocene folder")
    ensure_exists(PLIOCENE_DIR, "Pliocene folder")

    bio_ids = load_selected_bio_ids()
    points = read_occurrences()

    if len(points) < 10:
        raise ValueError("Too few occurrence points after cleaning.")

    samples_csv = export_samples(points)

    region = build_study_polygon()
    countries = load_countries()

    climate_dirs = {
        "current": CURRENT_DIR,
        "lgm": LGM_DIR,
        "pleistocene": PLEISTOCENE_DIR,
        "pliocene": PLIOCENE_DIR,
    }

    crop_dirs = {}
    ascii_dirs = {}

    for key, folder in climate_dirs.items():
        crop_dir = OUT_DIR / f"{key}_crop"
        ascii_dir = OUT_DIR / f"ascii_{key}"

        remove_and_recreate_dir(crop_dir)
        remove_and_recreate_dir(ascii_dir)

        layers = find_bio_layers(folder, bio_ids)

        print(f"\n{key.upper()} layers:")
        for bio_id, src in layers.items():
            print(f"BIO{bio_id}: {src}")

            tif_path = crop_dir / f"bio_{bio_id}.tif"
            asc_path = ascii_dir / f"bio_{bio_id}.asc"

            crop_raster(src, tif_path, region)
            tif_to_ascii(tif_path, asc_path)

        crop_dirs[key] = crop_dir
        ascii_dirs[key] = ascii_dir

    maxent_dirs = {
        "current": OUT_DIR / "maxent_current",
        "lgm": OUT_DIR / "maxent_lgm",
        "pleistocene": OUT_DIR / "maxent_pleistocene",
        "pliocene": OUT_DIR / "maxent_pliocene",
    }

    run_maxent(samples_csv, ascii_dirs["current"], ascii_dirs["current"], maxent_dirs["current"])
    run_maxent(samples_csv, ascii_dirs["current"], ascii_dirs["lgm"], maxent_dirs["lgm"])
    run_maxent(samples_csv, ascii_dirs["current"], ascii_dirs["pleistocene"], maxent_dirs["pleistocene"])
    run_maxent(samples_csv, ascii_dirs["current"], ascii_dirs["pliocene"], maxent_dirs["pliocene"])

    # IMPORTANT: keywords avoid picking same current raster for all periods
    predictions = {
        "current": find_prediction(maxent_dirs["current"]),
        "lgm": find_prediction(maxent_dirs["lgm"], "ascii_lgm"),
        "pleistocene": find_prediction(maxent_dirs["pleistocene"], "ascii_pleistocene"),
        "pliocene": find_prediction(maxent_dirs["pliocene"], "ascii_pliocene"),
    }

    pngs = {
        "current": OUT_DIR / f"{SPECIES_SAFE}_current_auto_stretch.png",
        "lgm": OUT_DIR / f"{SPECIES_SAFE}_lgm_auto_stretch.png",
        "pleistocene": OUT_DIR / f"{SPECIES_SAFE}_pleistocene_auto_stretch.png",
        "pliocene": OUT_DIR / f"{SPECIES_SAFE}_pliocene_auto_stretch.png",
    }

    plot_map(predictions["current"], countries, f"{SPECIES} - Current SDM", pngs["current"])
    plot_map(predictions["lgm"], countries, f"{SPECIES} - Last Glacial Maximum SDM", pngs["lgm"])
    plot_map(predictions["pleistocene"], countries, f"{SPECIES} - Pleistocene SDM", pngs["pleistocene"])
    plot_map(predictions["pliocene"], countries, f"{SPECIES} - Pliocene SDM", pngs["pliocene"])

    final_png = OUT_DIR / f"{SPECIES_SAFE}_4panel_past_corrected.png"
    make_four_panel(pngs, final_png)

    print("\nDONE ✅")
    print("Final corrected 4-panel saved here:")
    print(final_png)

if __name__ == "__main__":
    main()

Using selected BIO layers: [1, 4, 8, 9, 10, 12, 14, 15, 18, 19]
Clean occurrence points: 258

CURRENT layers:
BIO1: C:\Users\ual-laptop\Desktop\CAPSTONE\Time Period\Current\10min\bio_1.tif
BIO4: C:\Users\ual-laptop\Desktop\CAPSTONE\Time Period\Current\10min\bio_4.tif
BIO8: C:\Users\ual-laptop\Desktop\CAPSTONE\Time Period\Current\10min\bio_8.tif
BIO9: C:\Users\ual-laptop\Desktop\CAPSTONE\Time Period\Current\10min\bio_9.tif
BIO10: C:\Users\ual-laptop\Desktop\CAPSTONE\Time Period\Current\10min\bio_10.tif
BIO12: C:\Users\ual-laptop\Desktop\CAPSTONE\Time Period\Current\10min\bio_12.tif
BIO14: C:\Users\ual-laptop\Desktop\CAPSTONE\Time Period\Current\10min\bio_14.tif
BIO15: C:\Users\ual-laptop\Desktop\CAPSTONE\Time Period\Current\10min\bio_15.tif
BIO18: C:\Users\ual-laptop\Desktop\CAPSTONE\Time Period\Current\10min\bio_18.tif
BIO19: C:\Users\ual-laptop\Desktop\CAPSTONE\Time Period\Current\10min\bio_19.tif

LGM layers:
BIO1: C:\Users\ual-laptop\Desktop\CAPSTONE\Time Period\Last Glacial Maximum

In [6]:
# ============================================================
# FUTURE CORRELATION ANALYSIS FOR Semnopithecus entellus
# SPECIE_04_PD0588
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import matplotlib.pyplot as plt

SPECIES = "Semnopithecus entellus"
SPECIES_SAFE = SPECIES.replace(" ", "_")

BASE_DIR = Path(r"C:\Users\ual-laptop\Desktop\CAPSTONE")

GBIF_CSV = BASE_DIR / "data" / "gbif" / "Semnopithecus entellus.csv"
CURRENT_DIR = BASE_DIR / "Time Period" / "Current" / "10min"

OUT_DIR = BASE_DIR / "SPECIE_04_PD0588_Semnopithecus_entellus" / "02_results" / "future_correlation"
OUT_DIR.mkdir(parents=True, exist_ok=True)

COMMON_BIO_IDS = [1, 4, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
CORR_THRESHOLD = 0.75
STUDY_BOUNDS = (67.12, -12.6, 124.12, 39.35)

def read_occurrences(csv_path):
    df = pd.read_csv(csv_path)

    lon_col = next((c for c in ["decimalLongitude", "longitude", "lon", "x"] if c in df.columns), None)
    lat_col = next((c for c in ["decimalLatitude", "latitude", "lat", "y"] if c in df.columns), None)

    if lon_col is None or lat_col is None:
        raise ValueError(f"Could not find longitude/latitude columns. Available columns: {list(df.columns)}")

    df = df[[lon_col, lat_col]].dropna().copy()
    df.columns = ["lon", "lat"]

    df = df[
        (df["lon"] >= -180) & (df["lon"] <= 180) &
        (df["lat"] >= -90) & (df["lat"] <= 90)
    ].copy()

    return gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df["lon"], df["lat"]),
        crs="EPSG:4326"
    )

def remove_duplicate_coords(gdf):
    gdf = gdf.copy()
    gdf["x_round"] = gdf.geometry.x.round(6)
    gdf["y_round"] = gdf.geometry.y.round(6)
    gdf = gdf.drop_duplicates(subset=["x_round", "y_round"]).copy()
    return gdf.drop(columns=["x_round", "y_round"])

def clip_points_to_study_area(gdf, bounds):
    minx, miny, maxx, maxy = bounds
    return gdf[
        (gdf.geometry.x >= minx) &
        (gdf.geometry.x <= maxx) &
        (gdf.geometry.y >= miny) &
        (gdf.geometry.y <= maxy)
    ].copy()

def find_bio_layers(folder, bio_ids):
    tif_files = list(folder.glob("*.tif"))

    if not tif_files:
        raise FileNotFoundError(f"No .tif files found in {folder}")

    mapping = {}

    for bio_id in bio_ids:
        candidates = []

        for p in tif_files:
            stem = p.stem.lower().replace("-", "_")

            if (
                stem == f"bio_{bio_id}"
                or stem == f"bio{bio_id}"
                or stem == f"bio{bio_id:02d}"
                or f"bio_{bio_id}" in stem
                or f"bio{bio_id:02d}" in stem
                or f"bio{bio_id}" in stem
            ):
                candidates.append(p)

        if not candidates:
            raise FileNotFoundError(f"Could not find BIO{bio_id} in {folder}")

        mapping[bio_id] = sorted(candidates)[0]

    return mapping

def sample_raster_values(raster_paths, points_gdf):
    coords = [(geom.x, geom.y) for geom in points_gdf.geometry]
    data = pd.DataFrame(index=points_gdf.index)

    for bio_id, path in raster_paths.items():
        with rasterio.open(path) as src:
            vals = np.array([v[0] for v in src.sample(coords)], dtype=float)

            nodata = src.nodata
            if nodata is not None:
                vals = np.where(vals == nodata, np.nan, vals)

            vals = np.where(np.isinf(vals), np.nan, vals)
            data[f"bio_{bio_id}"] = vals

    return data

def find_high_correlation_pairs(corr_matrix, threshold):
    rows = []
    cols = list(corr_matrix.columns)

    for i in range(len(cols)):
        for j in range(i + 1, len(cols)):
            corr_val = corr_matrix.loc[cols[i], cols[j]]

            if pd.notna(corr_val) and abs(corr_val) > threshold:
                rows.append({
                    "Variable_1": cols[i],
                    "Variable_2": cols[j],
                    "Correlation": corr_val,
                    "Abs_Correlation": abs(corr_val)
                })

    if rows:
        return pd.DataFrame(rows).sort_values("Abs_Correlation", ascending=False)

    return pd.DataFrame(columns=["Variable_1", "Variable_2", "Correlation", "Abs_Correlation"])

def remove_correlated_variables(df, threshold):
    corr = df.corr(numeric_only=True).abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

    to_drop = []
    for col in upper.columns:
        if any(upper[col] > threshold):
            to_drop.append(col)

    retained = [c for c in df.columns if c not in to_drop]
    return retained, to_drop

def main():
    print("GBIF:", GBIF_CSV)
    print("CURRENT_DIR:", CURRENT_DIR)
    print("OUT_DIR:", OUT_DIR)

    occ = read_occurrences(GBIF_CSV)
    print("Raw occurrence points:", len(occ))

    occ = clip_points_to_study_area(occ, STUDY_BOUNDS)
    print("Occurrence points inside study area:", len(occ))

    occ = remove_duplicate_coords(occ)
    print("Points after duplicate removal:", len(occ))

    if len(occ) < 10:
        raise ValueError("Too few occurrence points after cleaning.")

    layers = find_bio_layers(CURRENT_DIR, COMMON_BIO_IDS)

    env = sample_raster_values(layers, occ)
    complete = env.dropna().copy()

    print("Complete raster samples:", len(complete))

    if complete.empty:
        raise ValueError("No valid raster values extracted at occurrence points.")

    corr_matrix = complete.corr(numeric_only=True)

    high_corr_pairs = find_high_correlation_pairs(corr_matrix, CORR_THRESHOLD)
    retained_vars, dropped_vars = remove_correlated_variables(complete, CORR_THRESHOLD)

    summary_df = pd.DataFrame({
        "Variable": list(corr_matrix.columns),
        "Status": ["Retained" if v in retained_vars else "Dropped" for v in corr_matrix.columns]
    })

    corr_matrix.to_excel(OUT_DIR / f"{SPECIES_SAFE}_future_correlation_matrix.xlsx")
    high_corr_pairs.to_excel(OUT_DIR / f"{SPECIES_SAFE}_future_high_correlation_pairs.xlsx", index=False)

    pd.DataFrame({"selected_variable": retained_vars}).to_csv(
        OUT_DIR / f"{SPECIES_SAFE}_future_selected_variables.csv",
        index=False
    )

    summary_df.to_excel(
        OUT_DIR / f"{SPECIES_SAFE}_future_variable_selection_summary.xlsx",
        index=False
    )

    heatmap_png = OUT_DIR / f"{SPECIES_SAFE}_future_correlation_heatmap.png"

    fig, ax = plt.subplots(figsize=(12, 10))
    im = ax.imshow(corr_matrix.values, interpolation="nearest")

    ax.set_xticks(range(len(corr_matrix.columns)))
    ax.set_yticks(range(len(corr_matrix.index)))
    ax.set_xticklabels(corr_matrix.columns, rotation=45, ha="right")
    ax.set_yticklabels(corr_matrix.index)

    for i in range(corr_matrix.shape[0]):
        for j in range(corr_matrix.shape[1]):
            ax.text(
                j,
                i,
                f"{corr_matrix.iloc[i, j]:.2f}",
                ha="center",
                va="center",
                fontsize=8
            )

    cbar = plt.colorbar(im, ax=ax, fraction=0.03, pad=0.03)
    cbar.set_label("Pearson correlation")

    ax.set_title(f"{SPECIES} - Future Variable Correlation Heatmap", fontsize=14)

    plt.tight_layout()
    plt.savefig(heatmap_png, dpi=300, bbox_inches="tight")
    plt.close()

    print("\nDONE ✅ FUTURE CORRELATION FILES CREATED")
    print("Retained variables:", retained_vars)
    print("Dropped variables:", dropped_vars)
    print("Output folder:", OUT_DIR)

if __name__ == "__main__":
    main()

GBIF: C:\Users\ual-laptop\Desktop\CAPSTONE\data\gbif\Semnopithecus entellus.csv
CURRENT_DIR: C:\Users\ual-laptop\Desktop\CAPSTONE\Time Period\Current\10min
OUT_DIR: C:\Users\ual-laptop\Desktop\CAPSTONE\SPECIE_04_PD0588_Semnopithecus_entellus\02_results\future_correlation
Raw occurrence points: 300
Occurrence points inside study area: 300
Points after duplicate removal: 258
Complete raster samples: 258

DONE ✅ FUTURE CORRELATION FILES CREATED
Retained variables: ['bio_1', 'bio_4', 'bio_8', 'bio_9', 'bio_10', 'bio_12', 'bio_14', 'bio_15', 'bio_18', 'bio_19']
Dropped variables: ['bio_11', 'bio_13', 'bio_16', 'bio_17']
Output folder: C:\Users\ual-laptop\Desktop\CAPSTONE\SPECIE_04_PD0588_Semnopithecus_entellus\02_results\future_correlation


In [7]:
# ============================================================
# FUTURE SDM PIPELINE FOR Semnopithecus entellus
# SPECIE_04_PD0588
# Uses future-correlation-selected variables automatically
# Produces clean SSP126 + SSP585 2-panel map
# ============================================================

import shutil
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.mask import mask
from rasterio.plot import plotting_extent
from shapely.geometry import box
import matplotlib.pyplot as plt

# ============================================================
# SETTINGS
# ============================================================

SPECIES = "Semnopithecus entellus"
SPECIES_SAFE = SPECIES.replace(" ", "_")

BASE_DIR = Path(r"C:\Users\ual-laptop\Desktop\CAPSTONE")
PROJECT_DIR = BASE_DIR / "SPECIE_04_PD0588_Semnopithecus_entellus"

OUT_DIR = PROJECT_DIR / "02_results" / "future"
OUT_DIR.mkdir(parents=True, exist_ok=True)

FUTURE_CORR_DIR = PROJECT_DIR / "02_results" / "future_correlation"
SELECTED_CSV = FUTURE_CORR_DIR / f"{SPECIES_SAFE}_future_selected_variables.csv"

JAVA_EXE = Path(r"C:\Program Files\Java\jdk-17\bin\java.exe")
MAXENT_JAR = BASE_DIR / "maxent" / "maxent.jar"

GBIF_CSV = BASE_DIR / "data" / "gbif" / "Semnopithecus entellus.csv"
COUNTRIES_SHP = BASE_DIR / "data" / "boundaries" / "ne_10m_admin_0_countries" / "ne_10m_admin_0_countries.shp"

CURRENT_DIR = BASE_DIR / "Time Period" / "Current" / "10min"

LOW_FUTURE_TIF = BASE_DIR / "Time Period" / "Future_CMIP6" / "MIROC6" / "wc2.1_2.5m_bioc_MIROC6_ssp126_2081-2100.tif"
HIGH_FUTURE_TIF = BASE_DIR / "Time Period" / "Future_CMIP6" / "MIROC6" / "wc2.1_2.5m_bioc_MIROC6_ssp585_2081-2100.tif"

STUDY_BOUNDS = (67.12, -12.6, 124.12, 39.35)

NODATA = -9999.0
FIG_DPI = 300
CMAP = "RdYlGn_r"   # green = low suitability, red = high suitability

# ============================================================
# HELPERS
# ============================================================

def ensure_exists(path, label):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"{label} not found: {path}")

def remove_and_recreate_dir(path):
    path = Path(path)
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)

def load_selected_bio_ids():
    ensure_exists(SELECTED_CSV, "Future selected variables CSV")

    df = pd.read_csv(SELECTED_CSV)
    col = "selected_variable" if "selected_variable" in df.columns else df.columns[0]

    bio_ids = []
    for v in df[col].dropna():
        v = str(v).strip().lower().replace("bio_", "").replace("bio", "")
        bio_ids.append(int(v))

    if not bio_ids:
        raise ValueError("No selected BIO layers found.")

    print("Using future selected BIO layers:", bio_ids)
    return bio_ids

def read_occurrences():
    df = pd.read_csv(GBIF_CSV)

    lon_col = next((c for c in ["decimalLongitude", "longitude", "lon", "x"] if c in df.columns), None)
    lat_col = next((c for c in ["decimalLatitude", "latitude", "lat", "y"] if c in df.columns), None)

    if lon_col is None or lat_col is None:
        raise ValueError(f"Longitude/latitude columns not found. Columns: {list(df.columns)}")

    df = df[[lon_col, lat_col]].dropna().copy()
    df.columns = ["lon", "lat"]

    gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df["lon"], df["lat"]),
        crs="EPSG:4326"
    )

    minx, miny, maxx, maxy = STUDY_BOUNDS
    gdf = gdf[
        (gdf.geometry.x >= minx) &
        (gdf.geometry.x <= maxx) &
        (gdf.geometry.y >= miny) &
        (gdf.geometry.y <= maxy)
    ].copy()

    gdf["x_round"] = gdf.geometry.x.round(6)
    gdf["y_round"] = gdf.geometry.y.round(6)
    gdf = gdf.drop_duplicates(subset=["x_round", "y_round"]).copy()
    gdf = gdf.drop(columns=["x_round", "y_round"])

    print("Clean occurrence points:", len(gdf))
    return gdf

def export_samples(points_gdf):
    samples_csv = OUT_DIR / f"{SPECIES_SAFE}_maxent_samples.csv"

    pd.DataFrame({
        "species": SPECIES,
        "longitude": points_gdf.geometry.x,
        "latitude": points_gdf.geometry.y
    }).to_csv(samples_csv, index=False)

    return samples_csv

def build_study_polygon():
    return gpd.GeoDataFrame(
        geometry=[box(*STUDY_BOUNDS)],
        crs="EPSG:4326"
    )

def load_countries():
    countries = gpd.read_file(COUNTRIES_SHP)
    if countries.crs is None:
        countries = countries.set_crs("EPSG:4326")
    return countries.to_crs("EPSG:4326")

def find_current_bio_layers(folder, bio_ids):
    files = list(Path(folder).glob("*.tif"))

    if not files:
        raise FileNotFoundError(f"No .tif files found in {folder}")

    mapping = {}

    for bio_id in bio_ids:
        candidates = []

        for f in files:
            stem = f.stem.lower().replace("-", "_")

            if (
                stem == f"bio_{bio_id}"
                or stem == f"bio{bio_id}"
                or stem == f"bio{bio_id:02d}"
                or f"bio_{bio_id}" in stem
                or f"bio{bio_id:02d}" in stem
                or f"bio{bio_id}" in stem
            ):
                candidates.append(f)

        if not candidates:
            raise FileNotFoundError(f"Could not find BIO{bio_id} in {folder}")

        mapping[bio_id] = sorted(candidates)[0]

    return mapping

def crop_current_raster(src_path, out_path, region):
    with rasterio.open(src_path) as src:
        img, transform = mask(src, list(region.geometry), crop=True, filled=False)

        arr = np.ma.array(img[0], copy=False).astype("float32")
        arr = arr.filled(NODATA).astype("float32")
        arr = np.where(np.isfinite(arr), arr, NODATA).astype("float32")

        meta = src.meta.copy()
        meta.update({
            "driver": "GTiff",
            "count": 1,
            "height": arr.shape[0],
            "width": arr.shape[1],
            "transform": transform,
            "dtype": "float32",
            "nodata": NODATA
        })

        out_path.parent.mkdir(parents=True, exist_ok=True)

        with rasterio.open(out_path, "w", **meta) as dst:
            dst.write(arr, 1)

def extract_future_band(src_tif, bio_id, region, out_tif):
    with rasterio.open(src_tif) as src:
        if bio_id < 1 or bio_id > src.count:
            raise ValueError(f"BIO{bio_id} requested, but future raster has only {src.count} bands.")

        img, transform = mask(
            src,
            list(region.geometry),
            crop=True,
            indexes=bio_id,
            filled=False
        )

        arr = np.ma.array(img, copy=False).astype("float32")
        arr = arr.filled(NODATA).astype("float32")
        arr = np.where(np.isfinite(arr), arr, NODATA).astype("float32")

        meta = src.meta.copy()
        meta.update({
            "driver": "GTiff",
            "count": 1,
            "height": arr.shape[0],
            "width": arr.shape[1],
            "transform": transform,
            "dtype": "float32",
            "nodata": NODATA
        })

        out_tif.parent.mkdir(parents=True, exist_ok=True)

        with rasterio.open(out_tif, "w", **meta) as dst:
            dst.write(arr, 1)

def tif_to_ascii(src_tif, out_asc):
    with rasterio.open(src_tif) as src:
        band = src.read(1, masked=True).astype("float32")

        arr = np.ma.array(band, copy=False).filled(NODATA).astype("float32")
        arr = np.where(np.isfinite(arr), arr, NODATA).astype("float32")

        transform = src.transform
        nrows, ncols = arr.shape

        xll = transform.c
        yll = transform.f + transform.e * nrows
        cellsize = transform.a

        out_asc.parent.mkdir(parents=True, exist_ok=True)

        with open(out_asc, "w") as f:
            f.write(f"ncols        {ncols}\n")
            f.write(f"nrows        {nrows}\n")
            f.write(f"xllcorner    {xll}\n")
            f.write(f"yllcorner    {yll}\n")
            f.write(f"cellsize     {cellsize}\n")
            f.write(f"NODATA_value {NODATA}\n")

            for row in arr:
                f.write(" ".join(f"{float(v):.6f}" for v in row) + "\n")

def validate_ascii_dir(folder):
    asc_files = list(Path(folder).glob("*.asc"))
    if not asc_files:
        raise ValueError(f"No ASCII files found in {folder}")

def run_maxent(samples_csv, env_dir, projection_dir, out_dir):
    validate_ascii_dir(env_dir)
    validate_ascii_dir(projection_dir)

    remove_and_recreate_dir(out_dir)

    cmd = [
        str(JAVA_EXE),
        "-Xmx4g",
        "-jar",
        str(MAXENT_JAR),
        f"samplesfile={samples_csv}",
        f"environmentallayers={env_dir}",
        f"projectionlayers={projection_dir}",
        f"outputdirectory={out_dir}",
        "outputformat=cloglog",
        "randomtestpoints=25",
        "maximumbackground=10000",
        "maximumiterations=5000",
        "autorun",
        "redoifexists",
        "novisible",
        "pictures=false",
        "responsecurves=false",
        "jackknife=false",
        "writebackgroundpredictions=false"
    ]

    print("\nRunning MaxEnt:")
    print(" ".join(cmd))

    result = subprocess.run(cmd, capture_output=True, text=True)

    print(result.stdout)
    print(result.stderr)

    if result.returncode != 0:
        raise RuntimeError("MaxEnt failed. Check output above.")

def find_prediction(folder, scenario_words):
    files = list(Path(folder).rglob("*.asc"))

    clean_files = [
        f for f in files
        if "clamping" not in f.name.lower()
        and "novel" not in f.name.lower()
        and "limiting" not in f.name.lower()
        and "background" not in f.name.lower()
    ]

    scenario_words = [w.lower() for w in scenario_words]

    matches = [
        f for f in clean_files
        if any(w in f.name.lower() for w in scenario_words)
    ]

    candidates = matches if matches else clean_files

    if not candidates:
        raise FileNotFoundError(f"No prediction raster found in {folder}")

    chosen = sorted(
        candidates,
        key=lambda p: (
            "median" not in p.name.lower(),
            "avg" not in p.name.lower(),
            p.name.lower()
        )
    )[0]

    print("Prediction chosen:", chosen)
    return chosen

def load_raster(path):
    with rasterio.open(path) as src:
        arr = src.read(1).astype(float)

        nodata = src.nodata
        if nodata is not None:
            arr[arr == nodata] = np.nan

        arr[np.isinf(arr)] = np.nan
        extent = plotting_extent(src)

    return arr, extent

def get_boundaries(countries):
    region = box(*STUDY_BOUNDS)
    region_gdf = gpd.GeoDataFrame(geometry=[region], crs="EPSG:4326")

    clipped = countries[countries.intersects(region)].copy()
    if clipped.empty:
        return clipped

    return gpd.clip(clipped, region_gdf)

def plot_future_map(arr, extent, countries, title, out_png, vmin, vmax):
    minx, miny, maxx, maxy = STUDY_BOUNDS
    boundaries = get_boundaries(countries)

    fig, ax = plt.subplots(figsize=(10, 6))

    im = ax.imshow(
        arr,
        extent=extent,
        origin="upper",
        cmap=CMAP,
        vmin=vmin,
        vmax=vmax
    )

    if not boundaries.empty:
        boundaries.boundary.plot(ax=ax, color="black", linewidth=0.5)

    ax.set_xlim(minx, maxx)
    ax.set_ylim(miny, maxy)

    ax.set_title(title, fontsize=15)
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")

    cbar = plt.colorbar(im, ax=ax, fraction=0.03, pad=0.03)
    cbar.set_label("Habitat suitability")

    plt.tight_layout()
    plt.savefig(out_png, dpi=FIG_DPI, bbox_inches="tight")
    plt.close()

def make_two_panel(ssp126_png, ssp585_png, final_png):
    imgs = [
        ("SSP126 Low Carbon Emission", plt.imread(ssp126_png)),
        ("SSP585 High Carbon Emission", plt.imread(ssp585_png)),
    ]

    fig, axes = plt.subplots(1, 2, figsize=(16, 7))

    for ax, (label, img_path) in zip(axes, imgs):
        ax.imshow(img_path)
        ax.axis("off")
        ax.set_title(label, fontsize=15)

    fig.suptitle(SPECIES, fontsize=20)

    plt.tight_layout()
    plt.savefig(final_png, dpi=FIG_DPI, bbox_inches="tight")
    plt.close()

# ============================================================
# MAIN
# ============================================================

def main():
    ensure_exists(SELECTED_CSV, "Future selected variables CSV")
    ensure_exists(JAVA_EXE, "java.exe")
    ensure_exists(MAXENT_JAR, "maxent.jar")
    ensure_exists(GBIF_CSV, "GBIF CSV")
    ensure_exists(COUNTRIES_SHP, "Countries shapefile")
    ensure_exists(CURRENT_DIR, "Current climate folder")
    ensure_exists(LOW_FUTURE_TIF, "SSP126 future raster")
    ensure_exists(HIGH_FUTURE_TIF, "SSP585 future raster")

    bio_ids = load_selected_bio_ids()
    points = read_occurrences()

    if len(points) < 10:
        raise ValueError("Too few occurrence points after cleaning.")

    samples_csv = export_samples(points)
    region = build_study_polygon()
    countries = load_countries()

    current_crop = OUT_DIR / "current_crop"
    ssp126_crop = OUT_DIR / "ssp126_crop"
    ssp585_crop = OUT_DIR / "ssp585_crop"

    ascii_current = OUT_DIR / "ascii_current"
    ascii_ssp126 = OUT_DIR / "ascii_ssp126"
    ascii_ssp585 = OUT_DIR / "ascii_ssp585"

    for folder in [current_crop, ssp126_crop, ssp585_crop, ascii_current, ascii_ssp126, ascii_ssp585]:
        remove_and_recreate_dir(folder)

    current_layers = find_current_bio_layers(CURRENT_DIR, bio_ids)

    for bio_id, src in current_layers.items():
        crop_current_raster(src, current_crop / f"bio_{bio_id}.tif", region)

    for bio_id in bio_ids:
        extract_future_band(LOW_FUTURE_TIF, bio_id, region, ssp126_crop / f"bio_{bio_id}.tif")
        extract_future_band(HIGH_FUTURE_TIF, bio_id, region, ssp585_crop / f"bio_{bio_id}.tif")

    for tif in current_crop.glob("*.tif"):
        tif_to_ascii(tif, ascii_current / tif.with_suffix(".asc").name)

    for tif in ssp126_crop.glob("*.tif"):
        tif_to_ascii(tif, ascii_ssp126 / tif.with_suffix(".asc").name)

    for tif in ssp585_crop.glob("*.tif"):
        tif_to_ascii(tif, ascii_ssp585 / tif.with_suffix(".asc").name)

    print("ASCII current:", len(list(ascii_current.glob("*.asc"))))
    print("ASCII SSP126:", len(list(ascii_ssp126.glob("*.asc"))))
    print("ASCII SSP585:", len(list(ascii_ssp585.glob("*.asc"))))

    maxent_current = OUT_DIR / "maxent_current"
    maxent_ssp126 = OUT_DIR / "maxent_future_ssp126"
    maxent_ssp585 = OUT_DIR / "maxent_future_ssp585"

    run_maxent(samples_csv, ascii_current, ascii_current, maxent_current)
    run_maxent(samples_csv, ascii_current, ascii_ssp126, maxent_ssp126)
    run_maxent(samples_csv, ascii_current, ascii_ssp585, maxent_ssp585)

    pred126 = find_prediction(maxent_ssp126, ["ssp126", "future_low", "low"])
    pred585 = find_prediction(maxent_ssp585, ["ssp585", "future_high", "high"])

    arr126, extent126 = load_raster(pred126)
    arr585, extent585 = load_raster(pred585)

    combined_values = np.concatenate([
        arr126[np.isfinite(arr126)],
        arr585[np.isfinite(arr585)]
    ])

    vmin = np.nanpercentile(combined_values, 2)
    vmax = np.nanpercentile(combined_values, 98)

    if not np.isfinite(vmin) or not np.isfinite(vmax) or vmin == vmax:
        vmin = np.nanmin(combined_values)
        vmax = np.nanmax(combined_values)

    ssp126_png = OUT_DIR / f"{SPECIES_SAFE}_SSP126_clean_gradient.png"
    ssp585_png = OUT_DIR / f"{SPECIES_SAFE}_SSP585_clean_gradient.png"
    final_png = OUT_DIR / f"{SPECIES_SAFE}_Future_2panel_clean_gradient.png"

    plot_future_map(
        arr126,
        extent126,
        countries,
        "SSP126 Low Carbon Emission",
        ssp126_png,
        vmin,
        vmax
    )

    plot_future_map(
        arr585,
        extent585,
        countries,
        "SSP585 High Carbon Emission",
        ssp585_png,
        vmin,
        vmax
    )

    make_two_panel(ssp126_png, ssp585_png, final_png)

    print("\nDONE ✅")
    print("Final future map saved here:")
    print(final_png)

if __name__ == "__main__":
    main()

Using future selected BIO layers: [1, 4, 8, 9, 10, 12, 14, 15, 18, 19]
Clean occurrence points: 258
ASCII current: 10
ASCII SSP126: 10
ASCII SSP585: 10

Running MaxEnt:
C:\Program Files\Java\jdk-17\bin\java.exe -Xmx4g -jar C:\Users\ual-laptop\Desktop\CAPSTONE\maxent\maxent.jar samplesfile=C:\Users\ual-laptop\Desktop\CAPSTONE\SPECIE_04_PD0588_Semnopithecus_entellus\02_results\future\Semnopithecus_entellus_maxent_samples.csv environmentallayers=C:\Users\ual-laptop\Desktop\CAPSTONE\SPECIE_04_PD0588_Semnopithecus_entellus\02_results\future\ascii_current projectionlayers=C:\Users\ual-laptop\Desktop\CAPSTONE\SPECIE_04_PD0588_Semnopithecus_entellus\02_results\future\ascii_current outputdirectory=C:\Users\ual-laptop\Desktop\CAPSTONE\SPECIE_04_PD0588_Semnopithecus_entellus\02_results\future\maxent_current outputformat=cloglog randomtestpoints=25 maximumbackground=10000 maximumiterations=5000 autorun redoifexists novisible pictures=false responsecurves=false jackknife=false writebackgroundpredic

In [1]:
# ============================================================
# FULL CORRECTED FUTURE SDM PIPELINE
# Semnopithecus entellus - SPECIE_04_PD0588
#
# FIX:
# Uses SAME BIO layers as past/current prediction
# ============================================================

import shutil
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.mask import mask
from rasterio.plot import plotting_extent
from shapely.geometry import box
import matplotlib.pyplot as plt


# ============================================================
# SETTINGS
# ============================================================

SPECIES = "Semnopithecus entellus"
SPECIES_SAFE = SPECIES.replace(" ", "_")

BASE_DIR = Path(r"C:\Users\ual-laptop\Desktop\CAPSTONE")
PROJECT_DIR = BASE_DIR / "SPECIE_04_PD0588_Semnopithecus_entellus"

OUT_DIR = PROJECT_DIR / "02_results" / "future_same_bio_layers"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# IMPORTANT: same selected BIO layers as past/current
CORR_DIR = PROJECT_DIR / "02_results" / "correlation"
SELECTED_CSV = CORR_DIR / f"{SPECIES_SAFE}_selected_variables.csv"

JAVA_EXE = Path(r"C:\Program Files\Java\jdk-17\bin\java.exe")
MAXENT_JAR = BASE_DIR / "maxent" / "maxent.jar"

GBIF_CSV = BASE_DIR / "data" / "gbif" / "Semnopithecus entellus.csv"

COUNTRIES_SHP = (
    BASE_DIR
    / "data"
    / "boundaries"
    / "ne_10m_admin_0_countries"
    / "ne_10m_admin_0_countries.shp"
)

CURRENT_DIR = BASE_DIR / "Time Period" / "Current" / "10min"

LOW_FUTURE_TIF = (
    BASE_DIR
    / "Time Period"
    / "Future_CMIP6"
    / "MIROC6"
    / "wc2.1_2.5m_bioc_MIROC6_ssp126_2081-2100.tif"
)

HIGH_FUTURE_TIF = (
    BASE_DIR
    / "Time Period"
    / "Future_CMIP6"
    / "MIROC6"
    / "wc2.1_2.5m_bioc_MIROC6_ssp585_2081-2100.tif"
)

STUDY_BOUNDS = (67.12, -12.6, 124.12, 39.35)

MAXENT_REPLICATES = 10
MAXENT_RANDOM_TEST = 25
MAXENT_BACKGROUND = 10000
MAXENT_ITERATIONS = 5000
MAXENT_OUTPUT_FORMAT = "cloglog"
REG_MULTIPLIER = 1.0

NODATA = -9999.0
FIG_DPI = 300
CMAP = "RdYlGn_r"

SSP126_PNG = OUT_DIR / f"{SPECIES_SAFE}_SSP126_future_clear.png"
SSP585_PNG = OUT_DIR / f"{SPECIES_SAFE}_SSP585_future_clear.png"
FINAL_2PANEL = OUT_DIR / f"{SPECIES_SAFE}_Future_2panel_same_BIO_layers.png"


# ============================================================
# HELPERS
# ============================================================

def ensure_exists(path, label):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"{label} not found: {path}")


def remove_and_recreate_dir(path):
    path = Path(path)
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)


def load_selected_bio_ids():
    ensure_exists(SELECTED_CSV, "Past/current selected variables CSV")

    df = pd.read_csv(SELECTED_CSV)

    if "selected_variable" in df.columns:
        col = "selected_variable"
    elif "kept_variable" in df.columns:
        col = "kept_variable"
    elif "Variable" in df.columns and "Status" in df.columns:
        df = df[df["Status"].astype(str).str.lower() == "retained"]
        col = "Variable"
    else:
        col = df.columns[0]

    bio_ids = []

    for v in df[col].dropna():
        v = str(v).strip().lower()
        v = v.replace("bio_", "").replace("bio", "")
        bio_ids.append(int(v))

    if not bio_ids:
        raise ValueError("No selected BIO layers found.")

    print("Using SAME BIO layers as past/current prediction:", bio_ids)
    return bio_ids


def read_occurrences():
    ensure_exists(GBIF_CSV, "GBIF CSV")

    df = pd.read_csv(GBIF_CSV)

    lon_col = next(
        (c for c in ["decimalLongitude", "longitude", "lon", "x"] if c in df.columns),
        None
    )

    lat_col = next(
        (c for c in ["decimalLatitude", "latitude", "lat", "y"] if c in df.columns),
        None
    )

    if lon_col is None or lat_col is None:
        raise ValueError(f"Longitude/latitude columns not found. Columns: {list(df.columns)}")

    df = df[[lon_col, lat_col]].dropna().copy()
    df.columns = ["lon", "lat"]

    gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df["lon"], df["lat"]),
        crs="EPSG:4326"
    )

    minx, miny, maxx, maxy = STUDY_BOUNDS

    gdf = gdf[
        (gdf.geometry.x >= minx) &
        (gdf.geometry.x <= maxx) &
        (gdf.geometry.y >= miny) &
        (gdf.geometry.y <= maxy)
    ].copy()

    gdf["x_round"] = gdf.geometry.x.round(6)
    gdf["y_round"] = gdf.geometry.y.round(6)

    gdf = gdf.drop_duplicates(subset=["x_round", "y_round"]).copy()
    gdf = gdf.drop(columns=["x_round", "y_round"])

    print("Clean occurrence points:", len(gdf))
    return gdf


def export_samples(points_gdf):
    samples_csv = OUT_DIR / f"{SPECIES_SAFE}_maxent_samples.csv"

    pd.DataFrame({
        "species": SPECIES,
        "longitude": points_gdf.geometry.x,
        "latitude": points_gdf.geometry.y
    }).to_csv(samples_csv, index=False)

    return samples_csv


def build_study_polygon():
    return gpd.GeoDataFrame(
        {"name": ["study_region"]},
        geometry=[box(*STUDY_BOUNDS)],
        crs="EPSG:4326"
    )


def load_countries():
    ensure_exists(COUNTRIES_SHP, "Countries shapefile")

    countries = gpd.read_file(COUNTRIES_SHP)

    if countries.crs is None:
        countries = countries.set_crs("EPSG:4326")

    return countries.to_crs("EPSG:4326")


def find_current_bio_layers(folder, bio_ids):
    folder = Path(folder)
    files = list(folder.glob("*.tif"))

    if not files:
        raise FileNotFoundError(f"No .tif files found in {folder}")

    mapping = {}

    for bio_id in bio_ids:
        candidates = []

        for f in files:
            stem = f.stem.lower().replace("-", "_")

            if (
                stem == f"bio_{bio_id}"
                or stem == f"bio{bio_id}"
                or stem == f"bio{bio_id:02d}"
                or f"bio_{bio_id}" in stem
                or f"bio{bio_id:02d}" in stem
                or f"bio{bio_id}" in stem
            ):
                candidates.append(f)

        if not candidates:
            raise FileNotFoundError(f"Could not find BIO{bio_id} in {folder}")

        mapping[bio_id] = sorted(candidates)[0]

    return mapping


def crop_single_raster(src_path, out_path, region):
    with rasterio.open(src_path) as src:
        img, transform = mask(
            src,
            list(region.geometry),
            crop=True,
            filled=False
        )

        arr = np.ma.array(img[0], copy=False).astype("float32")
        arr = arr.filled(NODATA).astype("float32")
        arr = np.where(np.isfinite(arr), arr, NODATA).astype("float32")

        meta = src.meta.copy()
        meta.update({
            "driver": "GTiff",
            "count": 1,
            "height": arr.shape[0],
            "width": arr.shape[1],
            "transform": transform,
            "dtype": "float32",
            "nodata": NODATA
        })

        out_path.parent.mkdir(parents=True, exist_ok=True)

        with rasterio.open(out_path, "w", **meta) as dst:
            dst.write(arr, 1)


def extract_future_band(src_tif, bio_id, region, out_tif):
    with rasterio.open(src_tif) as src:
        if bio_id < 1 or bio_id > src.count:
            raise ValueError(
                f"BIO{bio_id} requested, but future raster has only {src.count} bands."
            )

        img, transform = mask(
            src,
            list(region.geometry),
            crop=True,
            indexes=bio_id,
            filled=False
        )

        arr = np.ma.array(img, copy=False).astype("float32")
        arr = arr.filled(NODATA).astype("float32")
        arr = np.where(np.isfinite(arr), arr, NODATA).astype("float32")

        meta = src.meta.copy()
        meta.update({
            "driver": "GTiff",
            "count": 1,
            "height": arr.shape[0],
            "width": arr.shape[1],
            "transform": transform,
            "dtype": "float32",
            "nodata": NODATA
        })

        out_tif.parent.mkdir(parents=True, exist_ok=True)

        with rasterio.open(out_tif, "w", **meta) as dst:
            dst.write(arr, 1)


def tif_to_ascii(src_tif, out_asc):
    with rasterio.open(src_tif) as src:
        band = src.read(1, masked=True).astype("float32")

        arr = np.ma.array(band, copy=False).filled(NODATA).astype("float32")
        arr = np.where(np.isfinite(arr), arr, NODATA).astype("float32")

        transform = src.transform
        nrows, ncols = arr.shape

        xll = transform.c
        yll = transform.f + transform.e * nrows
        cellsize = transform.a

        out_asc.parent.mkdir(parents=True, exist_ok=True)

        with open(out_asc, "w") as f:
            f.write(f"ncols        {ncols}\n")
            f.write(f"nrows        {nrows}\n")
            f.write(f"xllcorner    {xll}\n")
            f.write(f"yllcorner    {yll}\n")
            f.write(f"cellsize     {cellsize}\n")
            f.write(f"NODATA_value {NODATA}\n")

            for row in arr:
                f.write(" ".join(f"{float(v):.6f}" for v in row) + "\n")


def validate_ascii_file(path):
    with open(path, "r") as f:
        lines = f.readlines()

    if len(lines) < 7:
        raise ValueError(f"ASCII file too short: {path}")

    header = lines[:6]
    data = lines[6:]

    ncols = int(header[0].split()[1])
    nrows = int(header[1].split()[1])

    if len(data) != nrows:
        raise ValueError(f"{path.name}: expected {nrows} rows, found {len(data)}")

    for i, row in enumerate(data, start=1):
        values = row.strip().split()

        if len(values) != ncols:
            raise ValueError(
                f"{path.name}: row {i} expected {ncols} columns, found {len(values)}"
            )

        if any(v.lower() == "nan" for v in values):
            raise ValueError(f"{path.name}: contains literal nan on row {i}")


def validate_ascii_directory(folder):
    folder = Path(folder)
    asc_files = list(folder.glob("*.asc"))

    if not asc_files:
        raise ValueError(f"No ASCII files found in {folder}")

    for asc in asc_files:
        validate_ascii_file(asc)


def run_maxent(samples_csv, env_dir, projection_dir, out_dir):
    ensure_exists(JAVA_EXE, "java.exe")
    ensure_exists(MAXENT_JAR, "maxent.jar")
    ensure_exists(samples_csv, "samples CSV")
    ensure_exists(env_dir, "environment directory")
    ensure_exists(projection_dir, "projection directory")

    validate_ascii_directory(env_dir)
    validate_ascii_directory(projection_dir)

    remove_and_recreate_dir(out_dir)

    cmd = [
        str(JAVA_EXE),
        "-Xmx4g",
        "-jar",
        str(MAXENT_JAR),
        f"samplesfile={samples_csv}",
        f"environmentallayers={env_dir}",
        f"projectionlayers={projection_dir}",
        f"outputdirectory={out_dir}",
        f"randomtestpoints={MAXENT_RANDOM_TEST}",
        f"maximumbackground={MAXENT_BACKGROUND}",
        f"replicates={MAXENT_REPLICATES}",
        "replicatetype=subsample",
        f"maximumiterations={MAXENT_ITERATIONS}",
        f"outputformat={MAXENT_OUTPUT_FORMAT}",
        f"betamultiplier={REG_MULTIPLIER}",
        "autofeature=true",
        "responsecurves=true",
        "jackknife=true",
        "pictures=true",
        "writebackgroundpredictions=false",
        "visible=false",
        "autorun",
        "redoifexists",
        "novisible"
    ]

    print("\nRunning MaxEnt:")
    print(" ".join(cmd))

    result = subprocess.run(cmd, capture_output=True, text=True)

    print(result.stdout)
    print(result.stderr)

    if result.returncode != 0:
        raise RuntimeError("MaxEnt failed. Check output above.")


def find_prediction(output_dir):
    output_dir = Path(output_dir)

    candidates = list(output_dir.glob("*.asc")) + list(output_dir.glob("*.tif"))

    candidates = [
        p for p in candidates
        if "clamping" not in p.name.lower()
        and "novel" not in p.name.lower()
        and "limiting" not in p.name.lower()
        and "background" not in p.name.lower()
    ]

    if not candidates:
        raise FileNotFoundError(f"No prediction raster found in {output_dir}")

    ranked = sorted(
        candidates,
        key=lambda p: (
            "median" not in p.name.lower(),
            "avg" not in p.name.lower(),
            p.name.lower()
        )
    )

    chosen = ranked[0]
    print("Prediction chosen:", chosen)
    return chosen


def load_raster(path):
    with rasterio.open(path) as src:
        arr = src.read(1).astype(float)

        nodata = src.nodata
        if nodata is not None:
            arr[arr == nodata] = np.nan

        arr[np.isinf(arr)] = np.nan
        extent = plotting_extent(src)

    return arr, extent


def get_boundaries(countries):
    region = box(*STUDY_BOUNDS)
    region_gdf = gpd.GeoDataFrame(geometry=[region], crs="EPSG:4326")

    clipped = countries[countries.intersects(region)].copy()

    if clipped.empty:
        return clipped

    return gpd.clip(clipped, region_gdf)


def normalize_for_clear_display(arr):
    finite = arr[np.isfinite(arr)]

    if finite.size == 0:
        raise ValueError("No valid raster values found for normalization.")

    vmin = np.nanpercentile(finite, 2)
    vmax = np.nanpercentile(finite, 98)

    if not np.isfinite(vmin) or not np.isfinite(vmax) or vmin == vmax:
        vmin = np.nanmin(finite)
        vmax = np.nanmax(finite)

    norm = (arr - vmin) / (vmax - vmin)
    norm = np.clip(norm, 0, 1)
    norm[~np.isfinite(arr)] = np.nan

    return norm


def plot_future_map(raster_path, countries, title, out_png):
    arr, extent = load_raster(raster_path)
    arr_norm = normalize_for_clear_display(arr)

    minx, miny, maxx, maxy = STUDY_BOUNDS
    boundaries = get_boundaries(countries)

    fig, ax = plt.subplots(figsize=(10, 6))

    im = ax.imshow(
        arr_norm,
        extent=extent,
        origin="upper",
        cmap=CMAP,
        vmin=0,
        vmax=1
    )

    if not boundaries.empty:
        boundaries.boundary.plot(ax=ax, color="black", linewidth=0.5)

    ax.set_xlim(minx, maxx)
    ax.set_ylim(miny, maxy)

    ax.set_title(title, fontsize=14)
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")

    cbar = plt.colorbar(im, ax=ax, fraction=0.03, pad=0.03)
    cbar.set_label("Relative habitat suitability")

    plt.tight_layout()
    plt.savefig(out_png, dpi=FIG_DPI, bbox_inches="tight")
    plt.close()


def make_two_panel(ssp126_png, ssp585_png, final_png):
    imgs = [
        ("SSP126 Low Carbon Emission", plt.imread(ssp126_png)),
        ("SSP585 High Carbon Emission", plt.imread(ssp585_png)),
    ]

    fig, axes = plt.subplots(1, 2, figsize=(16, 7))

    for ax, (label, img) in zip(axes, imgs):
        ax.imshow(img)
        ax.axis("off")
        ax.set_title(label, fontsize=15)

    fig.suptitle(f"{SPECIES} - Future SDM Projection", fontsize=20)

    plt.tight_layout()
    plt.savefig(final_png, dpi=FIG_DPI, bbox_inches="tight")
    plt.close()


# ============================================================
# MAIN
# ============================================================

def main():
    print("Output folder:", OUT_DIR)

    ensure_exists(SELECTED_CSV, "Past/current selected variables CSV")
    ensure_exists(JAVA_EXE, "java.exe")
    ensure_exists(MAXENT_JAR, "maxent.jar")
    ensure_exists(GBIF_CSV, "GBIF CSV")
    ensure_exists(COUNTRIES_SHP, "Countries shapefile")
    ensure_exists(CURRENT_DIR, "Current climate folder")
    ensure_exists(LOW_FUTURE_TIF, "SSP126 future raster")
    ensure_exists(HIGH_FUTURE_TIF, "SSP585 future raster")

    selected_bio_ids = load_selected_bio_ids()

    points = read_occurrences()

    if len(points) < 10:
        raise ValueError("Too few occurrence points after cleaning.")

    samples_csv = export_samples(points)

    region = build_study_polygon()
    countries = load_countries()

    current_crop = OUT_DIR / "current_crop"
    ssp126_crop = OUT_DIR / "ssp126_crop"
    ssp585_crop = OUT_DIR / "ssp585_crop"

    ascii_current = OUT_DIR / "ascii_current"
    ascii_ssp126 = OUT_DIR / "ascii_ssp126"
    ascii_ssp585 = OUT_DIR / "ascii_ssp585"

    for folder in [
        current_crop,
        ssp126_crop,
        ssp585_crop,
        ascii_current,
        ascii_ssp126,
        ascii_ssp585
    ]:
        remove_and_recreate_dir(folder)

    print("\nFinding current BIO layers...")
    current_layers = find_current_bio_layers(CURRENT_DIR, selected_bio_ids)

    print("\nCropping current BIO layers...")
    for bio_id, src in current_layers.items():
        crop_single_raster(
            src,
            current_crop / f"bio_{bio_id}.tif",
            region
        )

    print("\nExtracting same BIO bands from SSP126 and SSP585 future rasters...")
    for bio_id in selected_bio_ids:
        extract_future_band(
            LOW_FUTURE_TIF,
            bio_id,
            region,
            ssp126_crop / f"bio_{bio_id}.tif"
        )

        extract_future_band(
            HIGH_FUTURE_TIF,
            bio_id,
            region,
            ssp585_crop / f"bio_{bio_id}.tif"
        )

    print("\nConverting rasters to ASCII...")

    for tif in current_crop.glob("*.tif"):
        tif_to_ascii(tif, ascii_current / tif.with_suffix(".asc").name)

    for tif in ssp126_crop.glob("*.tif"):
        tif_to_ascii(tif, ascii_ssp126 / tif.with_suffix(".asc").name)

    for tif in ssp585_crop.glob("*.tif"):
        tif_to_ascii(tif, ascii_ssp585 / tif.with_suffix(".asc").name)

    print("\nASCII file counts:")
    print("Current:", len(list(ascii_current.glob("*.asc"))))
    print("SSP126:", len(list(ascii_ssp126.glob("*.asc"))))
    print("SSP585:", len(list(ascii_ssp585.glob("*.asc"))))

    maxent_current = OUT_DIR / "maxent_current"
    maxent_ssp126 = OUT_DIR / "maxent_future_ssp126"
    maxent_ssp585 = OUT_DIR / "maxent_future_ssp585"

    print("\nRunning current MaxEnt model...")
    run_maxent(samples_csv, ascii_current, ascii_current, maxent_current)

    print("\nProjecting model to SSP126 future climate...")
    run_maxent(samples_csv, ascii_current, ascii_ssp126, maxent_ssp126)

    print("\nProjecting model to SSP585 future climate...")
    run_maxent(samples_csv, ascii_current, ascii_ssp585, maxent_ssp585)

    print("\nFinding prediction rasters...")

    pred126 = find_prediction(maxent_ssp126)
    pred585 = find_prediction(maxent_ssp585)

    print("\nCreating future maps...")

    plot_future_map(
        pred126,
        countries,
        f"{SPECIES} - SSP126 Future SDM",
        SSP126_PNG
    )

    plot_future_map(
        pred585,
        countries,
        f"{SPECIES} - SSP585 Future SDM",
        SSP585_PNG
    )

    make_two_panel(
        SSP126_PNG,
        SSP585_PNG,
        FINAL_2PANEL
    )

    print("\n============================================================")
    print("DONE ✅")
    print("============================================================")
    print("Used SAME BIO layers as past/current prediction:")
    print(selected_bio_ids)
    print("\nFinal 2-panel future map saved here:")
    print(FINAL_2PANEL)


if __name__ == "__main__":
    main()

Output folder: C:\Users\ual-laptop\Desktop\CAPSTONE\SPECIE_04_PD0588_Semnopithecus_entellus\02_results\future_same_bio_layers
Using SAME BIO layers as past/current prediction: [1, 4, 8, 9, 10, 12, 14, 15, 18, 19]
Clean occurrence points: 258

Finding current BIO layers...

Cropping current BIO layers...

Extracting same BIO bands from SSP126 and SSP585 future rasters...

Converting rasters to ASCII...

ASCII file counts:
Current: 10
SSP126: 10
SSP585: 10

Running current MaxEnt model...

Running MaxEnt:
C:\Program Files\Java\jdk-17\bin\java.exe -Xmx4g -jar C:\Users\ual-laptop\Desktop\CAPSTONE\maxent\maxent.jar samplesfile=C:\Users\ual-laptop\Desktop\CAPSTONE\SPECIE_04_PD0588_Semnopithecus_entellus\02_results\future_same_bio_layers\Semnopithecus_entellus_maxent_samples.csv environmentallayers=C:\Users\ual-laptop\Desktop\CAPSTONE\SPECIE_04_PD0588_Semnopithecus_entellus\02_results\future_same_bio_layers\ascii_current projectionlayers=C:\Users\ual-laptop\Desktop\CAPSTONE\SPECIE_04_PD0588_S